# 学習済みモデルで句構造解析をする (Stanza)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_parsing/constituency_stanza.ipynb)

NLP クックブック 17-17 ／ 構文解析

文を名詞句・動詞句などの入れ子構造に解析したい。手書きの文法ではなく、ツリーバンクで学習したパーザを使いたい。

In [1]:
!pip install -q stanza

In [2]:
import stanza
from nltk import Tree

stanza.download("en", processors="tokenize,pos,constituency", verbose=False)
nlp = stanza.Pipeline("en", processors="tokenize,pos,constituency", verbose=False)

doc = nlp("The quick brown fox jumps over the lazy dog. I saw the man with the telescope.")
for sent in doc.sentences:
    print(sent.constituency)                                  # 括弧表記
    Tree.fromstring(str(sent.constituency)).pretty_print()    # NLTK の木に変換して表示

(ROOT (S (NP (DT The) (JJ quick) (JJ brown) (NN fox)) (VP (VBZ jumps) (PP (IN over) (NP (DT the) (JJ lazy) (NN dog)))) (. .)))
                     ROOT                          
                      |                             
                      S                            
       _______________|__________________________   
      |                         VP               | 
      |                _________|___             |  
      |               |             PP           | 
      |               |     ________|___         |  
      NP              |    |            NP       | 
  ____|__________     |    |     _______|____    |  
 DT   JJ    JJ   NN  VBZ   IN   DT      JJ   NN  . 
 |    |     |    |    |    |    |       |    |   |  
The quick brown fox jumps over the     lazy dog  . 

(ROOT (S (NP (PRP I)) (VP (VBD saw) (NP (DT the) (NN man)) (PP (IN with) (NP (DT the) (NN telescope)))) (. .)))
                ROOT                           
                 |            

In [3]:
# 名詞句と前置詞句を取り出す
def phrases(node, label):
    found = [" ".join(node.leaf_labels())] if node.label == label else []
    for child in node.children:
        found += phrases(child, label)
    return found

tree = doc.sentences[1].constituency
print("NP:", phrases(tree, "NP"))
print("PP:", phrases(tree, "PP"))

NP: ['I', 'the man', 'the telescope']
PP: ['with the telescope']


In [4]:
# 日本語の句構造解析
stanza.download("ja", processors="tokenize,pos,constituency", verbose=False)
nlp_ja = stanza.Pipeline("ja", processors="tokenize,pos,constituency", verbose=False)
tree = nlp_ja("太郎は図書館で借りた本を花子に渡した。").sentences[0].constituency
print(tree)
Tree.fromstring(str(tree)).pretty_print()

(ROOT (S (S (PP (BASENP (PROPN 太郎)) (ADP は)) (VP (PP (NP (VP (PP (BASENP (NOUN 図書) (NOUN 館)) (ADP で)) (S (VP (VERB 借り) (VP (AUX た))))) (NOUN 本)) (ADP を)) (VP (PP (BASENP (PROPN 花子)) (ADP に)) (VP (VERB 渡し) (VP (AUX た)))))) (PUNCT 。)))
                                                 ROOT                                          
                                                  |                                             
                                                  S                                            
                                               ___|_________________________________________    
                                              S                                             |  
         _____________________________________|___                                          |   
        |                                         VP                                        |  
        |                                      ___|_______________________                 

- 英語のモデルは Penn Treebank 形式のラベル（NP、VP、PP、SBAR …）を出力する。日本語のモデルは ALT (Asian Language Treebank) で学習されており、ラベル体系が違う。
- 「with the telescope」が動詞句と名詞句のどちらに付くかは、パーザが 1 つの解釈を選ぶ。手書き文法のレシピのように、すべての解釈は列挙されない。
- 依存構造と句構造は相互に変換できる情報を多く含むが、等価ではない。語と語の関係を見たいなら依存構造、句の範囲（スパン）が欲しいなら句構造が扱いやすい。